In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("../data/final_dataset.csv")


In [ ]:
df.head()

df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
print(df["price_sq_ft"].describe())
print(f"Median: {df["price_sq_ft"].median()}")
print(f"Skewness: {df["price_sq_ft"].skew()}")

sns.histplot(df["price_sq_ft"], bins=30, kde=True)
plt.title("Distribution of Price per Sq Ft")

### Price per Square Foot Observations

- **Distribution:** Right skewed distribution (skewness=3.27), with majority of listings clustered around $20-$40/sq ft.
- **Central Tendency:** Mean ($39.51/sq ft) > Median ($33) which confirms right skew
- **Typical Range:** 50% of listings fall between $27/sq ft and $45/sq ft
- **Outliers:** Multiple high priced listings with extreme anomolous outlier at $350/sq ft
- **Modelling:** Need to use log transformation to normalize data distribution for linear regression

In [ ]:
log_df = np.log(df["price_sq_ft"])
print(f"Skewness of log(price): {log_df.skew()}")

sns.histplot(log_df, bins=30)
plt.title("Distribution of Log Price per Sq Ft")
plt.xlabel("log_price_sq_ft")

In [ ]:
pd.set_option("display.float_format", lambda x: "%.2f" % x)
print(df["nearest_mtr_distance_m"].describe())
print(f"Skewness: {df["nearest_mtr_distance_m"].skew()}")

sns.histplot(df["nearest_mtr_distance_m"], bins=30)
plt.title("Distribution of Nearest Station Distance")

### Data Quality Check - MTR Distances

In [ ]:
error_df = df[df["nearest_mtr_distance_m"] > 5000].sort_values("nearest_mtr_distance_m", ascending=False)
error_df

### Data Quality Issue Found
- Multiple listings were encoded with the wrong geolocation coordinates (lat, lon) resulting in nearest station distances of up to 12,500km (almost the entire surface of the earth)
- Removing affected rows: geocoding api limit reached
- 32 rows affected

### **Fix Implemented:** Nearest MTR Distance Observations
- **Distribution:** Slightly right skewed (skewness=0.767), approximately normally distributed
- **Typical Range:** 156.7-361.14 meters to the nearest MTR station
- **Data Cleaning:** Dropped 32 erroneous rows
- **Modelling:** No changes needed

In [ ]:
numeric_cols = [
    "stations_within_1km",
    "num_gyms_within_100m",
    "num_cafes_within_100m",
    "num_restaurants_within_100m",
    "sq_ft"
    ]

for col in numeric_cols:
    print(f"{'='*50}")
    print(f"{col}")
    print(f"{'='*50}")
    print(f"{df[col].describe()}")
    print(f"Skewness: {df[col].skew():.3f}")

### Remaining Data Observations

#### Stations Within 1km
- **Distribution:** 

In [ ]:
df = df[df["sq_ft"] < 50000] # Remove listings above 50000 sq ft (outliers)
sns.histplot(df["sq_ft"], bins=50)
plt.title("Distribution of Square Footage")

In [ ]:
log_df = np.log(df["sq_ft"])
sns.histplot(log_df, bins=50)
plt.title("Log Distribution of Square Footage")
print(f"Skewness: {log_df.skew()}")

In [ ]:
print(df["district"].value_counts())
print(df.max())

In [ ]:
hk_island = ["CENTRAL", "WAN CHAI", "SHEUNG WAN", "CAUSEWAY BAY", "NORTH POINT", 
             "WONG CHUK HANG", "ADMIRALTY", "QUARRY BAY", "WESTERN / KENNEDY TOWN"]

kowloon = ["TSIM SHA TSUI", "KWUN TONG", "JORDAN", "MONG KOK / TAI KOK TSUI", 
           "HUNG HOM / TO KWA WAN", "KOWLOON BAY", "SAN PO KONG"]

new_territories = ["CHEUNG SHA WAN / LAI CHI KOK", "SHATIN / FO TAN", "KWAI CHUNG", "TSUEN WAN"]

df["region"] = df["district"].apply(lambda x: "HK Island" if x in hk_island else("Kowloon" if x in kowloon else "New Territories"))
print(df["region"].value_counts())

In [ ]:
df = df[df["region"] !=  "New Territories"] # Keep values that are not new territories
print(df["region"].value_counts())

### Note:
- Dropped listings in New Territories as it falls ouside of the scope of the analysis. The analysis will focus on business districts in Hong Kong.

In [ ]:
sns.boxplot(x="region", y="price_sq_ft", data=df)
plt.title("Price per Sq Ft by Region")

In [ ]:
sns.boxplot(x="district", y="price_sq_ft", data=df)
plt.title("Price Per Sq Ft by District")
plt.xticks(rotation=45, horizontalalignment="right")

In [ ]:
numeric_cols = [
    "stations_within_1km",
    "num_gyms_within_100m",
    "num_cafes_within_100m",
    "num_restaurants_within_100m",
    ]

for col in numeric_cols:
    sns.scatterplot(data=df, x=col, y="price_sq_ft")
    plt.title(f"Price Per Sq Ft by {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
df["near_mtr"] = (df["nearest_mtr_distance_m"] < 500).astype(int)
sns.boxplot(x="near_mtr", y="price_sq_ft", data=df)
print(df["near_mtr"].value_counts())

In [ ]:
df[["price_sq_ft", "nearest_mtr_distance_m", "num_cafes_within_100m", "num_gyms_within_100m", "sq_ft"]].corr()

In [ ]:
import statsmodels.api as sm

X = pd.get_dummies(df[["sq_ft", "nearest_mtr_distance_m", "num_cafes_within_100m", "district"]], columns=["district"], drop_first=True)
X = sm.add_constant(X)
y = df["price_sq_ft"]

model = sm.OLS(y, X).fit()
print(model.summary())